# Machine Learning for Applied Research

## 5: The Role of Machine Learning in the Applied Research Process

In this notebook, we will cover:
- The distinction between physics-based (mechanistic) and data-driven models
- Where hybrid, physics-informed approaches fit in
- A hands-on comparison of a simple physics-motivated rule against a data-driven model
- A tour of typical ML application patterns in engineering and the natural sciences

### Introduction

Notebook 1 introduced the idea of framing a research question as a machine learning task. Before going further, it is worth stepping back and asking a more fundamental question: when does it even make sense to reach for machine learning, as opposed to a model derived from established scientific or engineering theory? This notebook looks at that distinction directly, then surveys where ML actually gets used across engineering and the natural sciences, so that by the end, you can place your own research topic somewhere on that map.

### Physics-Based vs. Data-Driven Models

**Key distinction:**
- **Physics-based (mechanistic) model**: derived from established theory or first principles; requires known parameters; typically interpretable, and tends to extrapolate reliably within the regime the theory covers
- **Data-driven model**: learned from measured or simulated examples; makes few assumptions about the underlying mechanism; can capture complex or unknown relationships, but is only as trustworthy as the data it was trained on, and typically extrapolates poorly beyond it
- **Hybrid / physics-informed approaches**: a growing middle ground. For example, using a physics-based equation to generate a feature for a data-driven model, or constraining a data-driven model's outputs to obey a known physical law. Increasingly common in applied engineering research, though a deeper treatment is outside what we can cover in three days

For your own research: is there an established theory or formula that already partially describes the relationship you are interested in? If so, could it inform a data-driven model rather than being replaced by one?

### A Simple Comparison: Rule of Thumb vs. Fitted Model

We will use the Energy Efficiency dataset again, this time to make the physics-based vs. data-driven distinction concrete.

In [ ]:
!pip install ucimlrepo --quiet

from ucimlrepo import fetch_ucirepo
import pandas as pd

energy = fetch_ucirepo(id=242)
X = energy.data.features
y = energy.data.targets

X.columns = ["Relative Compactness",
             "Surface Area",
             "Wall Area",
             "Roof Area",
             "Overall Height",
             "Orientation",
             "Glazing Area",
             "Glazing Area Distribution"]

y.columns = ["Heating Load", "Cooling Load"]

target_col = y.columns[0]   # Heating Load
df = pd.concat([X, y[[target_col]]], axis=1)

print(f"Shape: {df.shape}")
df.head()

A simplified physics-motivated rule of thumb for a building's heat loss is that it scales roughly with exposed surface area: more surface, more heat lost to the outside. Let's turn that into an actual one-feature rule and see how far it gets us.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=[target_col]), df[target_col], test_size=0.25, random_state=42
)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def fit_proportional_rule(x_train, y_train):
    """Fit k in the rule y = k * x, using a regression through the origin."""
    return (x_train * y_train).sum() / (x_train ** 2).sum()

# A physics-motivated rule using only Surface Area, calibrated on the
# training data
k = fit_proportional_rule(X_train["Surface Area"], y_train)
rule_based_pred = k * X_test["Surface Area"]

print(f"Physics-motivated rule (Surface Area only) - Test RMSE: {rmse(y_test, rule_based_pred):.2f}")

#### 1.1 Exercises:
> (a) Using the same `X_train`, `X_test`, `y_train`, `y_test`, train a `LinearRegression` model using all available features (not just Surface Area).
>
> (b) Compute its test RMSE using the `rmse` function above, and compare it to the physics-motivated rule's RMSE.
>
> (c) In 1-2 sentences, explain why using all 8 features outperforms (or doesn't outperform) the single-feature rule - and name one situation where you might still prefer the simpler rule despite its higher error.

In [ ]:
# Your solution:




##### Solution:

In [ ]:
linreg = LinearRegression()
linreg.fit(X_train, y_train)
data_driven_pred = linreg.predict(X_test)

print(f"Data-driven model (all features) - Test RMSE: {rmse(y_test, data_driven_pred):.2f}")

# (c) The data-driven model outperforms the rule because heating load
# depends on more than just surface area (shape, height, glazing, and
# orientation all matter too), and the linear model can weigh all of them
# at once. You might still prefer the simpler rule when you need a rough
# estimate with no data and no computer at all.

#### 1.2 Exercises:
> (a) You've now seen Random Forests in Notebook 4. Fit a `RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)` on the same `X_train`/`y_train`, and compute its test RMSE.
>
> (b) Compare its RMSE to both the physics-motivated rule and the linear regression from Exercise 1.1. Does a more flexible model widen or narrow the gap with the simple one-feature rule?
>
> (c) In 1-2 sentences: does this suggest Heating Load is mostly explainable by a simple proportional relationship, or that meaningfully more is going on that only a flexible model captures?

In [ ]:
# Your solution:





##### Solution:

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
forest.fit(X_train, y_train)
forest_pred = forest.predict(X_test)

print(f"Physics-motivated rule           - Test RMSE: {rmse(y_test, rule_based_pred):.2f}")
print(f"Linear regression (all features) - Test RMSE: {rmse(y_test, data_driven_pred):.2f}")
print(f"Random Forest (all features)     - Test RMSE: {rmse(y_test, forest_pred):.2f}")

# (c) The Random Forest typically widens the gap even further beyond the
# linear model's improvement over the rule. That pattern suggests the
# relationship isn't just "more features, weighted linearly" - there are
# likely nonlinear interactions between features (e.g. how orientation
# matters differently depending on glazing area) that only a more
# flexible model can pick up on.

### Typical ML Applications in Engineering and the Natural Sciences

Machine learning shows up across engineering and the natural sciences in a fairly small number of recurring patterns. Recognizing which pattern your own research question fits is often the fastest way to figure out what kind of model and what kind of data you will need.

| Application pattern | What it does | Example |
|---|---|---|
| Predictive maintenance / anomaly detection | Flags unusual sensor behavior before a failure occurs | Vibration sensors predicting bearing wear in rotating machinery |
| Surrogate modeling | Approximates an expensive simulation or experiment | Predicting building energy load without re-running a full simulation |
| Remote sensing & environmental monitoring | Infers physical quantities from indirect signals | Estimating greenhouse gas concentrations from satellite spectra |
| Computer vision for inspection | Detects defects or classifies visual features automatically | Identifying manufacturing defects from camera images |
| Natural language processing | Extracts structure or meaning from text | Classifying user intent in a diagnostic chatbot |
| Forecasting | Predicts future values of a time-varying quantity | Predicting hourly demand from historical and weather data |
| Design-space exploration | Assesses or ranks many candidate designs quickly | Predicting a performance metric for thousands of conceptual design variants |

Which of these patterns is closest to your own research topic? Are there elements of more than one?

#### 1.3 Exercises:
> (a) For each of the five short scenarios below, decide whether it is closer to physics-based, data-driven, or a hybrid approach, and briefly justify your answer:
>
> 1. Predicting a material's fatigue life from its known composition, using an established engineering formula.
> 2. Classifying user intent in a chatbot from thousands of example conversations.
> 3. Estimating atmospheric gas concentration from satellite spectral measurements, using a model trained on ground-station measurements paired with a simplified radiative-transfer equation.
> 4. Predicting bike rental demand from years of historical usage and weather data.
> 5. Assessing thousands of conceptual building designs using a model trained on outputs from a validated engineering simulation tool.
>
> (b) For each scenario, name which application pattern from the table above it best matches.

In [ ]:
# Your solution:





##### Solution:


In [ ]:
# 1. Physics-based: a known engineering formula already links composition
#    to fatigue life, so no learning from data is required unless the
#    formula is unknown or needs correcting. It doesn't fit the table
#    directly because it's a classic mechanistic model, though it would
#    become a surrogate-modeling case if the formula were expensive to
#    evaluate many times.

# 2. Data-driven: there is no formula for "what a sentence means". The
#    model must learn purely from labeled examples. Pattern: natural
#    language processing.

# 3. Hybrid: a physics-based equation (radiative transfer) provides the
#    backbone, while a data-driven component corrects it using real
#    ground measurements. Pattern: remote sensing & environmental
#    monitoring.

# 4. Data-driven: demand is shaped by many interacting, hard-to-model
#    factors (habits, weather, events) with no governing equation.
#    Pattern: forecasting.

# 5. Data-driven, specifically surrogate modeling: the simulation tool IS
#    the physics-based ground truth here; the learned model approximates
#    it for speed. Pattern: design-space exploration / surrogate
#    modeling.

#### 1.4 Exercises (discussion):
> (a) In 2-3 sentences, describe your own research topic in terms of one (or a combination) of the application patterns in the table above.
>
> (b) Name one concrete feature or measurement from your own work that you'd feed into a model, and one thing about your data (small sample size, noise, missing labels, ...) that would need to be addressed before modeling it.
>
> (c) Compare notes in the chat: did you land on the same application pattern for similar-sounding research topics, or different ones? What made the difference?

In [ ]:
# Your solution:





##### Solution:


In [ ]:
# There is no single correct answer here, 
# this is a thinking/discussion exercise.
# A useful way to check your own answer: could you name the target
# variable and at least two candidate features for your scenario, the
# way Notebook 1's problem_framing template asked for? If not, that is
# often a sign the application pattern chosen is still a bit too vague to
# act on yet.

#### Challenge 1:
> Repeat the rule-of-thumb comparison using `Roof Area` and then `Glazing Area` instead of `Surface Area` (still calibrated with `fit_proportional_rule`). How do their test RMSEs compare to the Surface Area rule and to the data-driven model from Exercise 1.1?

In [ ]:
# Your solution:




##### Solution:


In [ ]:
for feature in ["Roof Area", "Glazing Area"]:
    k_alt = fit_proportional_rule(X_train[feature], y_train)
    alt_pred = k_alt * X_test[feature]
    print(f"Rule based on {feature:20s} - Test RMSE: {rmse(y_test, alt_pred):.2f}")

#### Challenge 2:
> Refit the physics-motivated rule and the Random Forest using only the first 100 rows of `X_train`/`y_train` (`X_train.iloc[:100]`, `y_train.iloc[:100]`), then evaluate both on the full `X_test`. How does the gap between the rule and the Random Forest change when data is scarce, compared to using the full training set? 
> What does this suggest about when a simple physics-based rule might actually be the safer choice, not because it's more accurate, but because it needs less data to begin with?

In [ ]:
# Your solution:





##### Solution:


In [ ]:
X_small = X_train.iloc[:100]
y_small = y_train.iloc[:100]

k_small = fit_proportional_rule(X_small["Surface Area"], y_small)
rule_small_pred = k_small * X_test["Surface Area"]

forest_small = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
forest_small.fit(X_small, y_small)
forest_small_pred = forest_small.predict(X_test)

print(f"Rule (100 rows)          - Test RMSE: {rmse(y_test, rule_small_pred):.2f}")
print(f"Random Forest (100 rows) - Test RMSE: {rmse(y_test, forest_small_pred):.2f}")

# With only 100 training rows, the Random Forest's advantage over the
# rule usually shrinks noticeably compared to the full-data comparison
# in Exercise 1.2: a flexible model needs enough examples to learn its
# extra flexibility safely, and with too little data it can overfit
# instead. A physics-based rule doesn't need to "learn" its form at all,
# only calibrate a single constant, which is why it can be the more
# reliable choice specifically in very small-data settings.

## Conclusion

This notebook looked at the "why" and "where" of machine learning in applied research:

- **Physics-based vs. data-driven models**: two different starting points, each with its own strengths, plus a growing middle ground of hybrid approaches
- **A quantitative comparison**: a simple one-feature rule vs. increasingly flexible data-driven models (linear regression, then Random Forest), showing concretely what added flexibility buys you (and costs), when data is scarce
- **A map of typical ML applications**: predictive maintenance, surrogate modeling, remote sensing, computer vision, NLP, forecasting, and design-space exploration. These are patterns that show up again and again across engineering and the natural sciences, including your own!